# TC-PhysXAI: Physics-Grounded XAI for Tropical Cyclone Track Prediction

**Author:** Nguyen Hong Gam  
**Paper:** TC-XFlow + TC-PhysXAI (ESWA 2025 + Extension)  
**Environment:** Kaggle GPU (P100/T4)

---

## Overview

| Module | Contribution | Description |
|--------|-------------|-------------|
| Module 1 | **C1** | Temporal attribution via climatological perturbation (ΔCRPS per 6h window) |
| Module 2 | **C2** | Integrated Gradients on FNO3D + physical object detection |
| Module 2 | **C3** | H-mean-F fidelity metric = 0.4·P + 0.4·R + 0.2·U |
| Module 3 | **M3** | DTW K-medoids scenario clustering + latent analogue retrieval |
| C4 | **C4** | FM integration-path attribution |
| C5 | **C5** | Variance decomposition: V_total = V_noise + V_context |
| C6 | **C6** | FM counterfactual via projected gradient on context space |
| **NEW** | **C7** | Segment-level temporal XAI + AOPC faithfulness |
| **NEW** | **C8** | Track Feature Attribution: speed/curvature/heading → ERA5 drivers |

> Notebook này chỉ chứa lệnh chạy. Toàn bộ code nằm trong GitHub repo.

---

## 1. Setup

1. Upload `TC-PhysXAI_Bundle.zip` thành Kaggle Dataset (Add Data → Upload), gắn vào notebook.
2. Gắn thêm 2 dataset: `tc-ofm` (kaggle1234uitvn) và `new-checkpoint` (gmnguynhng).
3. Bật GPU, chạy lần lượt các cell.

In [ ]:
%%bash
# 1.1 Tìm bundle (zip hoặc thư mục đã giải nén) trong /kaggle/input rồi cài vào /kaggle/working
set -e
SRC=$(find /kaggle/input -maxdepth 6 -type d -name TC-PhysXAI 2>/dev/null | head -1)
if [ -n "$SRC" ]; then
  ROOT=$(dirname "$SRC")
else
  Z=$(find /kaggle/input /kaggle/working -maxdepth 6 -name 'TC-PhysXAI_Bundle.zip' 2>/dev/null | head -1)
  if [ -z "$Z" ]; then echo "ERROR: khong tim thay TC-PhysXAI_Bundle.zip trong /kaggle/input. Hay Add Data -> upload zip."; exit 1; fi
  rm -rf /tmp/bundle && mkdir -p /tmp/bundle && unzip -q -o "$Z" -d /tmp/bundle
  ROOT=$(dirname "$(find /tmp/bundle -maxdepth 3 -type d -name TC-PhysXAI | head -1)")
fi
echo "Bundle root: $ROOT"
rm -rf /kaggle/working/TC-PhysXAI /kaggle/working/Tropical_Typhoon_Architecture
cp -r "$ROOT/TC-PhysXAI" /kaggle/working/TC-PhysXAI
cp -r "$ROOT/Tropical_Typhoon_Architecture" /kaggle/working/Tropical_Typhoon_Architecture
mkdir -p /kaggle/working/TC-PhysXAI/results
ls /kaggle/working/TC-PhysXAI /kaggle/working/Tropical_Typhoon_Architecture


In [ ]:
# 1.1b (TUỲ CHỌN – cần Internet ON) Cập nhật Model TC-XFlow mới nhất từ GitHub
!rm -rf /tmp/txf && git clone --depth 1 https://github.com/Hana-ai-18/Tropical_Typhoon_Architecture /tmp/txf && rm -rf /kaggle/working/Tropical_Typhoon_Architecture && cp -r /tmp/txf /kaggle/working/Tropical_Typhoon_Architecture && rm -rf /kaggle/working/Tropical_Typhoon_Architecture/.git && ls /kaggle/working/Tropical_Typhoon_Architecture/Model

In [ ]:
# 1.2 Kiểm tra môi trường (PhysXAI, Model TC-XFlow, checkpoint, dataset, GPU)
!python /kaggle/working/TC-PhysXAI/Scripts/check_env.py

## 2. Run Experiments

- **Option A (Recommended):** Multi-seed — E1 chạy 3 seeds, E2–E8 chạy seed0
- **Option B:** Single-seed — tất cả chạy seed0

> Thời gian T4: Option A ≈ 45–90 phút | Option B ≈ 30–60 phút

In [ ]:
# ── Option A (RECOMMENDED): Multi-Seed E1–E8 ────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/run_multiseed.py \
    --checkpoints \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed1_new_ver.pth \
        /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed2_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 \
    --experiments E1 E2 E3 E4 E5 E6 E7 E8

In [ ]:
# ── Option B: Single-Seed E1–E8 ─────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Experiments/run_all_experiments.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 \
    --experiments E1 E2 E3 E4 E5 E6 E7 E8

## 3. Chạy từng Experiment

In [ ]:
# ── E1
!python /kaggle/working/TC-PhysXAI/Experiments/E1_comparison_baselines.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E2
!python /kaggle/working/TC-PhysXAI/Experiments/E2_recurvature_attribution.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 30 --gpu 0

In [ ]:
# ── E3
!python /kaggle/working/TC-PhysXAI/Experiments/E3_rapid_intensification.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 15 --gpu 0

In [ ]:
# ── E4
!python /kaggle/working/TC-PhysXAI/Experiments/E4_multimodal_consistency.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E5
!python /kaggle/working/TC-PhysXAI/Experiments/E5_fidelity_benchmark.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E6
!python /kaggle/working/TC-PhysXAI/Experiments/E6_counterfactual_scenarios.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 15 --gpu 0

In [ ]:
# ── E7
!python /kaggle/working/TC-PhysXAI/Experiments/E7_adaptive_segment_xai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 20 --gpu 0

In [ ]:
# ── E8
!python /kaggle/working/TC-PhysXAI/Experiments/E8_track_feature_attribution.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --n_storms 20 --K 10 --gpu 0

## 4. Full Pipeline

In [ ]:
# ── Full: 20 storms ─────────────────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 20 --K 30 --run_experiments

In [ ]:
# ── Quick test: 1 storm ─────────────────────────────────────────────────────
!python /kaggle/working/TC-PhysXAI/Scripts/run_physxai.py \
    --checkpoint   /kaggle/input/datasets/gmnguynhng/new-checkpoint/best_model_fm_seed0_new_ver.pth \
    --dataset_root /kaggle/input/datasets/kaggle1234uitvn/tc-ofm \
    --output_dir   /kaggle/working/TC-PhysXAI/results \
    --gpu 0 --n_storms 1 --K 10

## 5. Xem kết quả

In [ ]:
!find /kaggle/working/TC-PhysXAI/results -type f | sort

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/TC-PhysXAI/results/**/*.png', recursive=True))[:6]:
    print(p); display(Image(p))

In [ ]:
!cat /kaggle/working/TC-PhysXAI/results/E1_multiseed_summary.json 2>/dev/null || \
 cat /kaggle/working/TC-PhysXAI/results/E1_summary.json 2>/dev/null || \
 echo 'Chưa có kết quả — chạy Section 2 trước.'

---

## Appendix: ERA5 Channel Reference

| Index | Variable | Level |
|-------|----------|-------|
| 0–3 | GPH | 200/500/850/925 hPa |
| 4–7 | U-wind | 200/500/850/925 hPa |
| 8–11 | V-wind | 200/500/850/925 hPa |
| 12 | SST | Surface |

```bibtex
@article{nguyen2025tcxflow,
  title={TC-XFlow},
  author={Nguyen, Hong Gam and others},
  journal={Expert Systems with Applications}, year={2025}
}
```